# 10 · Análise da multitarefa: MT × redes de tarefa única, pipeline de 4 classes e custo

**Sessão P2 · CPU** (Colab sem GPU, ou localmente). Lê `resultados/multitarefa` (MT, do `09`), `resultados/ablacao_cabeca` e `resultados/mlsd_cabeca` (redes de tarefa única do P1) e `resultados/multitarefa/calibracao_mt.json` (tempo de inferência, do `09`). Confere a integridade, **recalcula as métricas a partir dos `pred_teste.csv`**, faz o bootstrap pareado e aplica a regra abaixo.

## Regra pré-registrada (plano P2 de 14/10/2026, com os ajustes de 19/10/2026 das notas do P1, §7–9; fixada antes de qualquer rodada da MT)

* **Redes:** MT-KAN (`mt_kan`) e MT-MLP (`mt_mlp`), do `09`. ST-KAN = cabeça (d) do P1 nas duas tarefas (`resnet152__kan` + `mlsd__resnet152__kan`); ST-MLP = cabeça (b) do P1 (`resnet152__mlp128` + `mlsd__resnet152__mlp128`). Mesmas dobras, mesma receita, mesmas sementes (42, 43, 44).
* **Métrica primária:** AUC de cada tarefa, de probabilidades (na distância, OvR macro só nas entradas). F1 macro é secundária. A acurácia da distância vem com a nota de que prever sempre "distância" já dá 0,854.
* **Pareamento:** cada contraste usa só as sementes que os dois lados têm completas (5 dobras). Por imagem, a probabilidade é a média dessas sementes, e o IC 95 % sai de um bootstrap pareado de **casos** (2.000 reamostragens, semente 0) sobre o OOF, que tem as mesmas imagens dos dois lados (conferido). Por dobra, a métrica é a média das sementes.
* **Contrastes primários: MT × ST com a mesma cabeça** (MT-KAN × ST-KAN e MT-MLP × ST-MLP), em cada tarefa:
  * **melhora** = MT vence em ≥ 4/5 dobras **e** o IC 95 % do ΔAUC fica acima de 0;
  * **pior** = MT vence em ≤ 1/5 dobras **e** o IC 95 % do ΔAUC fica abaixo de 0 (mesmo que dentro da margem, o que a frase diz);
  * **não inferior** = nem melhora nem pior, com o limite inferior do IC 95 % do ΔAUC acima de **−0,02** (a margem do P3);
  * senão, **inconclusivo** (plausível na distância, que tem ~8 imagens de encostado por dobra).
* **Contraste secundário: KAN × MLP dentro da MT** (MT-KAN × MT-MLP), com a mesma regra. Descritivo: ST-KAN × ST-MLP (o (d) × (b) do P1, nas mesmas sementes).
* **Pipeline de 4 classes** {saída, encostado, curta, distância} (secundária, sem regra de decisão): no par ST, a rede E/S decide e, se ela disser "entrada", vale a rede de distância; na MT, as duas cabeças da mesma rede. F1 macro, com IC por bootstrap de casos.
* **Custo:** parâmetros; tempo de treino por dobra (15·t_F1 + 45·t_F2, nas rodadas da L4) da MT contra a soma das duas ST; tempo de inferência na L4 para obter as duas saídas (1 forward da MT × 2 do par ST, lotes 1 e 32). "Mais rápida" só se a diferença passar do ruído (média + dp < média − dp do rival).
* **Proibido:** falar em *parameter efficiency* da cabeça KAN (o argumento aqui é **uma rede no lugar de duas**, com os parâmetros contados); fazer afirmação específica sobre encostado (16 casos); comparar com 0,908 (TCC antigo) ou 0,893 (artigo do SIBGRAPI); dizer que a KAN spline é "a referência que funciona". O resultado entra **só no TCC** (artigo e pôster mantêm a multitarefa como trabalho futuro). A base é de Lira et al. (2025, IJLM) e não tem licença de redistribuição.
* **Sanidade:** acurácia E/S por dobra entre 0,80 e 0,89; F1 macro da distância > 0,31 (prever sempre "distância" dá 0,307).

Saídas em `resultados/`: `tabela_multitarefa.{csv,tex}`, `contrastes_multitarefa.{csv,tex}`, `pipeline_multitarefa.csv`, `confusao_multitarefa.csv`, `decisao_multitarefa.json` e `frases_multitarefa.md`. Em `resultados/multitarefa/<config>/`: `runs.csv`, `resumo.json` e `oof.csv`.

In [1]:
# ==== Configuração ====
NO_COLAB = True    # False para rodar localmente (aponte os caminhos abaixo para cópias locais)
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"
DIR_CODIGO = DIR_DRIVE                       # protocolo_comum.py e multitarefa.py (localmente: notebooks/)
DIR_RESULTADOS = f"{DIR_DRIVE}/resultados"   # localmente: uma cópia da pasta resultados/ do Drive
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_PLUGIN_ESPERADO = "b9841e0f8c0192bf67df96f318e384659a96659c5eda08737dae740f612c1814"   # multitarefa.py
SHA_MLSD_ESPERADO = "e2138640a83ef7587200ad6e051262b6a49e5b60027f6b4a61407fd1f61889f7"     # configs_mlsd.py (as ST de distância do P1 rodaram com ele)
N_BOOT = 2000
MARGEM_NI = -0.02   # margem de não inferioridade pré-registrada (ΔAUC)

In [2]:
if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
sys.path.insert(0, DIR_CODIGO)
import protocolo_comum as pc
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, "protocolo_comum.py diferente do publicado"
assert pc.sha256_arquivo(f"{DIR_CODIGO}/multitarefa.py") == SHA_PLUGIN_ESPERADO, "multitarefa.py diferente do publicado"
pc.carregar_plugin(f"{DIR_CODIGO}/multitarefa.py")
mt = sys.modules["multitarefa"]
R = Path(DIR_RESULTADOS)
DIR_EXP = R / mt.EXPERIMENTO
TAREFAS = ("es", "dist")
TIPOS = tuple(mt.CABECA)                              # ("kan", "mlp")
IDS = ["st_kan", "mt_kan", "st_mlp", "mt_mlp"]
ROTULO = {"es": "entrada/saída", "dist": "distância do disparo (só entradas)"}
TASK_EN = {"es": "entry/exit classification", "dist": "shooting-distance classification (entries only)"}
CABECA_EN = {"kan": "spline-KAN heads", "mlp": "MLP-128 heads"}
CABECA_PT = {"kan": "cabeças KAN spline", "mlp": "cabeças MLP-128"}
NOMES = {"st_kan": "Two single-task ResNet-152, KAN heads (P1 (d))", "mt_kan": "One multitask ResNet-152, KAN heads",
         "st_mlp": "Two single-task ResNet-152, MLP-128 heads (P1 (b))", "mt_mlp": "One multitask ResNet-152, MLP-128 heads"}
# id -> config, por tarefa (na MT a mesma config responde as duas tarefas)
CFG = {t: {**{f"st_{k}": mt.CONFIGS_ST[k][t] for k in TIPOS}, **{f"mt_{k}": mt.CONFIGS[k] for k in TIPOS}} for t in TAREFAS}
ID = {t: {v: k for k, v in CFG[t].items()} for t in TAREFAS}
SEMENTES_PLANO = list(mt.SEMENTES)

def ler(exp):
    r = pc.ler_resultados(R, exp)
    return r[~r["teste_rapido"].astype(bool)].copy() if len(r) else r

brutos = {"st_es": ler(mt.EXPERIMENTOS_ST["es"]), "st_dist": ler(mt.EXPERIMENTOS_ST["dist"]), "mt": ler(mt.EXPERIMENTO)}
runs = {}
for t in TAREFAS:
    partes = [x for x in (brutos[f"st_{t}"], brutos["mt"]) if len(x)]
    r = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=["config", "semente", "dobra"])
    r = r[r["config"].isin(list(ID[t]))].copy()
    r["id"] = r["config"].map(ID[t])
    runs[t] = r.reset_index(drop=True)
    print(f"{ROTULO[t]}: {len(r)} unidades")
    if len(r):
        display(r.groupby(["id", "semente"]).size().rename("dobras").unstack("semente"))
arq_b = DIR_EXP / "calibracao_mt.json"
bench = json.loads(arq_b.read_text()) if arq_b.exists() else {}
if not bench:
    print("⚠️ Sem calibracao_mt.json (rode o 09 com TESTE_RAPIDO = True): as colunas de inferência ficam vazias.")
elif bench.get("plugin_sha256") != SHA_PLUGIN_ESPERADO:
    print("⚠️ calibracao_mt.json é de outra versão do multitarefa.py.")

entrada/saída: 60 unidades


semente,42,43,44
id,,,
mt_kan,5,5,5
mt_mlp,5,5,5
st_kan,5,5,5
st_mlp,5,5,5


distância do disparo (só entradas): 60 unidades


semente,42,43,44
id,,,
mt_kan,5,5,5
mt_mlp,5,5,5
st_kan,5,5,5
st_mlp,5,5,5


## 1. Integridade (hashes, GPU, sementes completas, métricas recalculadas das predições, mesmas imagens)

In [4]:
def prob_cols(t):
    return [f"p_{t}_{c}" for c in range(pc.TAREFAS[t].n_classes)]

def auc_tarefa(y, P):
    if P.shape[1] == 2:
        return roc_auc_score(y, P[:, 1])
    return roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(P.shape[1])))

problemas, avisos = [], []
cfgs_p2 = set(CFG["es"].values()) | set(CFG["dist"].values())
todos = pd.concat([x for x in brutos.values() if len(x)], ignore_index=True)
todos = todos[todos["config"].isin(list(cfgs_p2))].copy()
todos["reserva"] = ~todos["gpu"].fillna("").str.contains("L4")
for t in TAREFAS:
    runs[t]["reserva"] = ~runs[t]["gpu"].fillna("").str.contains("L4")
for col in ("protocolo_sha256", "dobras_sha256", "precisao"):
    vals = todos[col].dropna().unique()
    if len(vals) != 1:
        problemas.append(f"{col} não é único: {vals}")
if len(todos) and todos["protocolo_sha256"].iloc[0] != SHA_PROTOCOLO_ESPERADO:
    problemas.append("SHA do protocolo diferente do publicado")
for _, r in todos.iterrows():
    pl = json.loads(r["plugins"])
    if r["config"] in mt.CONFIGS.values() and pl.get("multitarefa") != SHA_PLUGIN_ESPERADO:
        problemas.append(f"{r['config']} s{r['semente']} d{r['dobra']}: rodou com outro multitarefa.py")
    if r["config"].startswith("mlsd__") and pl.get("configs_mlsd") != SHA_MLSD_ESPERADO:
        problemas.append(f"{r['config']} s{r['semente']} d{r['dobra']}: rodou com outro configs_mlsd.py")
    if int(r["params_total"]) != mt.PARAMETROS_TOTAIS[r["config"]]:
        problemas.append(f"{r['config']} s{r['semente']} d{r['dobra']}: {r['params_total']} parâmetros (esperado {mt.PARAMETROS_TOTAIS[r['config']]})")
for f in (DIR_EXP.rglob("falha_nan.json") if DIR_EXP.exists() else []):
    problemas.append(f"perda NaN registrada em {f.parent}")
for p in sorted(R.iterdir()):
    if p.is_dir() and p.name.startswith(mt.EXPERIMENTO) and p.name != mt.EXPERIMENTO:
        avisos.append(f"pasta {p.name!r} em resultados/: confira no Drive se não é uma cópia duplicada")
if todos["reserva"].any():
    print("† unidades fora da L4:")
    display(todos.loc[todos.reserva].groupby(["experimento", "config", "semente"]).size())

# sementes completas (5 dobras) por config; o resto fica fora da análise, com aviso
SEM, incompletas = {t: {} for t in TAREFAS}, []
for t in TAREFAS:
    keep = []
    for (cfg, s), g in runs[t].groupby(["config", "semente"]):
        if sorted(g["dobra"]) == list(range(pc.N_DOBRAS)):
            SEM[t].setdefault(cfg, []).append(int(s)); keep.append(g)
        else:
            incompletas.append(f"{ROTULO[t]} {ID[t][cfg]} s{s}: dobras {sorted(g['dobra'])}")
    runs[t] = pd.concat(keep, ignore_index=True) if keep else runs[t].iloc[0:0]
    for i, cfg in CFG[t].items():
        falta = sorted(set(SEMENTES_PLANO) - set(SEM[t].get(cfg, [])))
        if falta:
            avisos.append(f"{ROTULO[t]} {i}: sementes ainda não concluídas {falta}")
if incompletas:
    avisos.append("sementes incompletas (fora da análise): " + "; ".join(sorted(set(incompletas))))

# predições por unidade (todas as imagens de teste) e pico de memória do treino
PRED, MEM = {}, {}
def pred_unidade(r):
    k = (r["config"], int(r["semente"]), int(r["dobra"]))
    if k not in PRED:
        d = Path(r["caminho"])
        PRED[k] = pd.read_csv(d / "pred_teste.csv", dtype={"case_id": str})
        h = pd.read_csv(d / "historico.csv") if (d / "historico.csv").exists() else pd.DataFrame()
        MEM[k] = float(h["mem_pico_gb"].max()) if "mem_pico_gb" in h.columns else float("nan")
    return PRED[k]

OOF = {}
for t in TAREFAS:
    col = pc.TAREFAS[t].coluna
    partes = []
    for _, r in runs[t].iterrows():
        p = pred_unidade(r)
        v = p[p[col] >= 0]
        y, P = v[col].to_numpy(), v[prob_cols(t)].to_numpy()
        pred = P.argmax(1)
        rec = {f"{t}_acc": accuracy_score(y, pred), f"{t}_auc": auc_tarefa(y, P),
               f"{t}_rec_macro": recall_score(y, pred, average="macro"),
               f"{t}_prec_macro": precision_score(y, pred, average="macro", zero_division=0),
               f"{t}_f1_macro": f1_score(y, pred, average="macro"),
               f"{t}_acc_balanceada": balanced_accuracy_score(y, pred)}
        for k, val in rec.items():
            if abs(val - r[k]) > 1e-9:
                problemas.append(f"{r['config']} s{r['semente']} d{r['dobra']}: {k} recalculado {val} ≠ {r[k]}")
        if abs(r[f"{t}_auc"] - r[f"{t}_rec_macro"]) < 1e-12:
            avisos.append(f"{r['config']} s{r['semente']} d{r['dobra']}: AUC = revocação macro (modelo degenerado?)")
        partes.append(v.assign(config=r["config"], id=r["id"], semente=int(r["semente"]), dobra=int(r["dobra"])))
    OOF[t] = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

# mesmas imagens de teste em todas as redes, dobra a dobra (ST-E/S, ST-distância e MT); probabilidades completas
ref = {}
for (cfg, s, d), p in PRED.items():
    ref.setdefault(d, frozenset(p["rel_path"]))
    if frozenset(p["rel_path"]) != ref[d]:
        problemas.append(f"{cfg} s{s} d{d}: imagens de teste diferentes das outras redes")
    if p[[c for c in p.columns if c.startswith("p_")]].isna().any().any():
        problemas.append(f"{cfg} s{s} d{d}: probabilidades faltando")

# sanidade (plano P2)
fora = runs["es"][(runs["es"]["es_acc"] < 0.80) | (runs["es"]["es_acc"] > 0.89)]
if len(fora):
    avisos.append("acurácia E/S fora de [0,80; 0,89]: " + ", ".join(
        f"{a} s{b} d{c} ({x:.3f})" for a, b, c, x in fora[["id", "semente", "dobra", "es_acc"]].itertuples(index=False)))
baixa = runs["dist"][runs["dist"]["dist_f1_macro"] <= 0.31]
if len(baixa):
    avisos.append("F1 macro da distância ≤ 0,31 (perto de prever sempre a classe majoritária): " + ", ".join(
        f"{a} s{b} d{c} ({x:.3f})" for a, b, c, x in baixa[["id", "semente", "dobra", "dist_f1_macro"]].itertuples(index=False)))
print("Problemas:", problemas or "nenhum")
print("Avisos:", avisos or "nenhum")
assert not problemas

Problemas: nenhum
Avisos: ['acurácia E/S fora de [0,80; 0,89]: mt_kan s42 d3 (0.792), mt_kan s43 d3 (0.797), mt_kan s44 d2 (0.793), mt_kan s44 d3 (0.796), mt_mlp s42 d3 (0.796)']


In [5]:
# resultados/multitarefa/<config>/{runs.csv, resumo.json, oof.csv} (refeitos aqui, com todas as abas juntas)
METR_MT = [f"{t}_{m}" for t in TAREFAS for m in ("acc", "prec_macro", "rec_macro", "f1_macro", "acc_balanceada", "auc")]
for cfg in mt.CONFIGS.values():
    r = brutos["mt"][brutos["mt"]["config"] == cfg] if len(brutos["mt"]) else brutos["mt"]
    if r.empty:
        continue
    d = DIR_EXP / cfg
    r.to_csv(d / "runs.csv", index=False)
    resumo = {f"s{sem}": {"n_dobras": int(len(g)),
                          **{m: {"media": float(g[m].mean()), "dp": float(g[m].std(ddof=1)) if len(g) > 1 else None} for m in METR_MT},
                          "treino_min_por_dobra": float(g["treino_min"].mean()), "params_total": int(g["params_total"].iloc[0]),
                          "gpus": sorted(g["gpu"].dropna().unique().tolist())}
              for sem, g in r.groupby("semente")}
    pc.gravar_json_atomico(resumo, d / "resumo.json")
    pd.concat([pd.read_csv(Path(x["caminho"]) / "pred_teste.csv", dtype={"case_id": str}).assign(config=x["config"], semente=x["semente"], dobra=x["dobra"])
               for _, x in r.iterrows()]).to_csv(d / "oof.csv", index=False)
    print(f"{cfg}: {len(r)} unidades -> runs.csv, resumo.json, oof.csv")

mt_kan: 15 unidades -> runs.csv, resumo.json, oof.csv
mt_mlp: 15 unidades -> runs.csv, resumo.json, oof.csv


## 2. Média ± dp entre dobras (cada dobra = média das sementes completas)

In [6]:
def resumir(t):
    r = runs[t].copy()
    if r.empty:
        return pd.DataFrame()
    metr = [f"{t}_{m}" for m in ("acc", "prec_macro", "rec_macro", "f1_macro", "acc_balanceada", "auc")]
    por_dobra = r.groupby(["id", "dobra"])[metr].mean().reset_index()
    agg = por_dobra.groupby("id")[metr].agg(["mean", "std", "count"])
    agg.columns = [f"{a}_{b}" for a, b in agg.columns]
    info = r.groupby("id").agg(params_total=("params_total", "first"), reserva=("reserva", "any"))
    info["sementes"] = r.groupby("id")["semente"].apply(lambda s: sorted(set(int(x) for x in s)))
    auc_oof = {}
    for i, g in OOF[t].groupby("id"):
        o = g.groupby(["rel_path", pc.TAREFAS[t].coluna])[prob_cols(t)].mean().reset_index()
        auc_oof[i] = auc_tarefa(o[pc.TAREFAS[t].coluna].to_numpy(), o[prob_cols(t)].to_numpy())
    out = agg.join(info)
    out["auc_oof"] = pd.Series(auc_oof)
    return out.loc[[i for i in IDS if i in out.index]]

RES = {t: resumir(t) for t in TAREFAS}
for t in TAREFAS:
    if len(RES[t]):
        print(ROTULO[t])
        display(RES[t][[f"{t}_auc_mean", f"{t}_auc_std", "auc_oof", f"{t}_f1_macro_mean", f"{t}_f1_macro_std",
                        f"{t}_acc_mean", f"{t}_acc_balanceada_mean", "sementes"]].round(4))
print(f"Distância: prever sempre a classe majoritária dá acurácia {mt.ACC_MAJORITARIA_DIST:.3f} e F1 macro "
      f"{2 * mt.ACC_MAJORITARIA_DIST / (1 + mt.ACC_MAJORITARIA_DIST) / 3:.3f}.")

entrada/saída


,es_auc_mean,es_auc_std,auc_oof,es_f1_macro_mean,es_f1_macro_std,es_acc_mean,es_acc_balanceada_mean,sementes
id,,,,,,,,
st_kan,0.9050,0.0134,0.9061,0.8060,0.0134,0.8456,0.8159,"[42, 43, 44]"
mt_kan,0.8942,0.0137,0.8954,0.7839,0.0223,0.8197,0.8093,"[42, 43, 44]"
st_mlp,0.9050,0.0137,0.9062,0.8058,0.0148,0.8457,0.8151,"[42, 43, 44]"
mt_mlp,0.8954,0.0140,0.8967,0.7873,0.0171,0.8231,0.8119,"[42, 43, 44]"


distância do disparo (só entradas)


,dist_auc_mean,dist_auc_std,auc_oof,dist_f1_macro_mean,dist_f1_macro_std,dist_acc_mean,dist_acc_balanceada_mean,sementes
id,,,,,,,,
st_kan,0.7977,0.0575,0.8054,0.4771,0.0451,0.8129,0.5145,"[42, 43, 44]"
mt_kan,0.8047,0.0653,0.8127,0.4825,0.0578,0.8243,0.5110,"[42, 43, 44]"
st_mlp,0.8015,0.0621,0.8106,0.4821,0.0505,0.8127,0.5202,"[42, 43, 44]"
mt_mlp,0.8059,0.0707,0.8148,0.4921,0.0627,0.8249,0.5196,"[42, 43, 44]"


Distância: prever sempre a classe majoritária dá acurácia 0.854 e F1 macro 0.307.


## 3. Contrastes pareados (bootstrap de casos) e regra pré-registrada

In [7]:
def sn(x, nd=3):
    # com sinal; se arredondar para zero, mostra mais casas (para o sinal não sumir nem enganar)
    for n in (nd, nd + 1, nd + 2):
        s = f"{x:+.{n}f}"
        if float(s) != 0:
            return s
    return f"{0:.{nd}f}"

def f1_macro_rapido(y, pred, k):
    cmx = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cmx); den = 2 * tp + (cmx.sum(0) - tp) + (cmx.sum(1) - tp)
    return float(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0).mean())

def oof_medio(t, cfg, sementes):
    col = pc.TAREFAS[t].coluna
    o = OOF[t][(OOF[t].config == cfg) & OOF[t].semente.isin(sementes)]
    return o.groupby(["rel_path", "case_id", col, "dobra"])[prob_cols(t)].mean().reset_index().set_index("rel_path")

def bootstrap_casos(casos, estatistica, n_boot=N_BOOT, semente=0):
    # reamostra CASOS (com todas as imagens de cada um); devolve as estatísticas válidas
    uc = np.unique(casos)
    idx_caso = {c: np.flatnonzero(casos == c) for c in uc}
    rng = np.random.default_rng(semente)
    out = []
    for _ in range(n_boot):
        am = np.concatenate([idx_caso[c] for c in rng.choice(uc, size=len(uc), replace=True)])
        try:
            out.append(estatistica(am))
        except ValueError:      # reamostra sem alguma classe
            continue
    return np.asarray(out)

def comparar(t, id_a, id_b, n_boot=N_BOOT, semente=0):
    # Δ (a − b): AUC e F1 macro no OOF (média das sementes comuns), IC 95 % por bootstrap de casos, e por dobra.
    cfg_a, cfg_b = CFG[t][id_a], CFG[t][id_b]
    sem = sorted(set(SEM[t].get(cfg_a, [])) & set(SEM[t].get(cfg_b, [])))
    if not sem:
        return None
    col, k = pc.TAREFAS[t].coluna, pc.TAREFAS[t].n_classes
    a, b = oof_medio(t, cfg_a, sem), oof_medio(t, cfg_b, sem)
    assert set(a.index) == set(b.index), f"{id_a} × {id_b}: OOF com imagens diferentes"
    b = b.loc[a.index]
    y = a[col].to_numpy().astype(int)
    Pa, Pb = a[prob_cols(t)].to_numpy(), b[prob_cols(t)].to_numpy()
    pa, pb = Pa.argmax(1), Pb.argmax(1)
    d_auc = auc_tarefa(y, Pa) - auc_tarefa(y, Pb)
    d_f1 = f1_macro_rapido(y, pa, k) - f1_macro_rapido(y, pb, k)
    bs = bootstrap_casos(a["case_id"].to_numpy(), lambda am: (auc_tarefa(y[am], Pa[am]) - auc_tarefa(y[am], Pb[am]),
                                                               f1_macro_rapido(y[am], pa[am], k) - f1_macro_rapido(y[am], pb[am], k)),
                         n_boot, semente)
    r = runs[t][runs[t].semente.isin(sem)]
    pdob = r.groupby(["id", "dobra"])[[f"{t}_auc", f"{t}_f1_macro"]].mean()
    dd = (pdob.loc[id_a] - pdob.loc[id_b]).dropna()
    return {"tarefa": t, "a": id_a, "b": id_b, "sementes": sem,
            "delta_auc_oof": float(d_auc), "ic95_auc": [float(x) for x in np.percentile(bs[:, 0], [2.5, 97.5])],
            "delta_f1_oof": float(d_f1), "ic95_f1": [float(x) for x in np.percentile(bs[:, 1], [2.5, 97.5])],
            "n_boot_validos": int(len(bs)),
            "delta_auc_dobras": {int(i): round(float(v), 4) for i, v in dd[f"{t}_auc"].items()},
            "delta_f1_dobras": {int(i): round(float(v), 4) for i, v in dd[f"{t}_f1_macro"].items()},
            "media_delta_auc_dobras": float(dd[f"{t}_auc"].mean()),
            "dp_delta_auc_dobras": float(dd[f"{t}_auc"].std(ddof=1)) if len(dd) > 1 else None,
            "dobras_a_vence": int((dd[f"{t}_auc"] > 0).sum()), "dobras_a_nao_perde": int((dd[f"{t}_auc"] >= 0).sum()),
            "n_dobras": int(len(dd))}

def avaliar(c):
    sup = c["dobras_a_vence"] >= 4 and c["ic95_auc"][0] > 0
    inf = c["dobras_a_vence"] <= 1 and c["ic95_auc"][1] < 0
    ni = c["ic95_auc"][0] > MARGEM_NI
    c.update(superior=bool(sup), inferior=bool(inf), dentro_da_margem=bool(ni))
    c["desfecho"] = "melhora" if sup else ("pior" if inf else ("nao_inferior" if ni else "inconclusivo"))
    return c

PLANO = [("mt_kan", "st_kan", "primário"), ("mt_mlp", "st_mlp", "primário"),
         ("mt_kan", "mt_mlp", "secundário"), ("st_kan", "st_mlp", "descritivo")]
COMP, linhas = {}, []
for t in TAREFAS:
    for a, b, papel in PLANO:
        c = comparar(t, a, b)
        if c is None:
            continue
        c = avaliar(c); c["papel"] = papel
        COMP[f"{t}__{a}_vs_{b}"] = c
        linhas.append({"tarefa": t, "contraste": f"{a} − {b}", "papel": papel, "sementes": ",".join(map(str, c["sementes"])),
                       "ΔAUC OOF": round(c["delta_auc_oof"], 4), "IC95 AUC": [round(x, 4) for x in c["ic95_auc"]],
                       "ΔF1 OOF": round(c["delta_f1_oof"], 4), "IC95 F1": [round(x, 4) for x in c["ic95_f1"]],
                       "a vence (dobras)": f"{c['dobras_a_vence']}/{c['n_dobras']}",
                       "a não perde (dobras)": f"{c['dobras_a_nao_perde']}/{c['n_dobras']}",
                       "desfecho": c["desfecho"] if papel != "descritivo" else "descritivo"})
TAB_C = pd.DataFrame(linhas)
display(TAB_C)
for k, c in COMP.items():
    if c["papel"] == "descritivo":
        continue
    print(f"{ROTULO[c['tarefa']]} — {c['a']} × {c['b']} ({c['papel']}): {c['desfecho'].upper()} "
          f"(ΔAUC {c['delta_auc_oof']:+.4f}, IC95 [{c['ic95_auc'][0]:+.4f}, {c['ic95_auc'][1]:+.4f}]; "
          f"{c['a']} vence em {c['dobras_a_vence']}/{c['n_dobras']} dobras; sementes {c['sementes']})")

,tarefa,contraste,papel,sementes,ΔAUC OOF,IC95 AUC,ΔF1 OOF,IC95 F1,a vence (dobras),a não perde (dobras),desfecho
0,es,mt_kan − st_kan,primário,"42,43,44",-0.0107,"[-0.0154, -0.006]",-0.0225,"[-0.0343, -0.0113]",0/5,0/5,pior
1,es,mt_mlp − st_mlp,primário,"42,43,44",-0.0095,"[-0.0141, -0.0051]",-0.0221,"[-0.0342, -0.0101]",0/5,0/5,pior
2,es,mt_kan − mt_mlp,secundário,"42,43,44",-0.0013,"[-0.0029, 0.0004]",-0.0019,"[-0.0077, 0.0034]",2/5,2/5,nao_inferior
3,es,st_kan − st_mlp,descritivo,"42,43,44",-0.0001,"[-0.0015, 0.0011]",-0.0015,"[-0.0063, 0.0032]",3/5,3/5,descritivo
4,dist,mt_kan − st_kan,primário,"42,43,44",0.0073,"[-0.003, 0.0176]",-0.0016,"[-0.029, 0.027]",3/5,3/5,nao_inferior
5,dist,mt_mlp − st_mlp,primário,"42,43,44",0.0043,"[-0.0064, 0.0144]",0.0073,"[-0.0213, 0.035]",3/5,3/5,nao_inferior
6,dist,mt_kan − mt_mlp,secundário,"42,43,44",-0.0022,"[-0.0085, 0.0033]",-0.0075,"[-0.0145, -0.0013]",2/5,2/5,nao_inferior
7,dist,st_kan − st_mlp,descritivo,"42,43,44",-0.0052,"[-0.0118, 0.0026]",0.0013,"[-0.007, 0.0097]",2/5,2/5,descritivo


entrada/saída — mt_kan × st_kan (primário): PIOR (ΔAUC -0.0107, IC95 [-0.0154, -0.0060]; mt_kan vence em 0/5 dobras; sementes [42, 43, 44])
entrada/saída — mt_mlp × st_mlp (primário): PIOR (ΔAUC -0.0095, IC95 [-0.0141, -0.0051]; mt_mlp vence em 0/5 dobras; sementes [42, 43, 44])
entrada/saída — mt_kan × mt_mlp (secundário): NAO_INFERIOR (ΔAUC -0.0013, IC95 [-0.0029, +0.0004]; mt_kan vence em 2/5 dobras; sementes [42, 43, 44])
distância do disparo (só entradas) — mt_kan × st_kan (primário): NAO_INFERIOR (ΔAUC +0.0073, IC95 [-0.0030, +0.0176]; mt_kan vence em 3/5 dobras; sementes [42, 43, 44])
distância do disparo (só entradas) — mt_mlp × st_mlp (primário): NAO_INFERIOR (ΔAUC +0.0043, IC95 [-0.0064, +0.0144]; mt_mlp vence em 3/5 dobras; sementes [42, 43, 44])
distância do disparo (só entradas) — mt_kan × mt_mlp (secundário): NAO_INFERIOR (ΔAUC -0.0022, IC95 [-0.0085, +0.0033]; mt_kan vence em 2/5 dobras; sementes [42, 43, 44])


## 4. Distância por classe e pipeline de 4 classes (secundários, descritivos)

Por classe e matriz de confusão no OOF (média das sementes): são ~8 imagens de encostado por dobra, então **nada de afirmação específica sobre encostado**. No pipeline, o par ST junta, por `rel_path`, o `pred_teste.csv` da rede E/S e o da rede de distância da mesma semente e dobra (o teste da distância do P1 é a dobra inteira, então as saídas também têm `p_dist_*`).

In [8]:
NOMES_DIST = pc.TAREFAS["dist"].nomes_classes
NOMES4 = ("saida",) + tuple(NOMES_DIST)
linhas_cm = []

def linhas_confusao(analise, i, y, pred, nomes):
    k = len(nomes)
    cmx = confusion_matrix(y, pred, labels=list(range(k)))
    rec = recall_score(y, pred, average=None, labels=list(range(k)), zero_division=0)
    f1c = f1_score(y, pred, average=None, labels=list(range(k)), zero_division=0)
    return [{"analise": analise, "id": i, "classe_real": nomes[a], "n": int(cmx[a].sum()), "recall": float(rec[a]),
             "f1": float(f1c[a]), **{f"pred_{nomes[b]}": int(cmx[a, b]) for b in range(k)}} for a in range(k)]

for i in IDS:
    cfg = CFG["dist"][i]
    if not SEM["dist"].get(cfg):
        continue
    o = oof_medio("dist", cfg, SEM["dist"][cfg])
    linhas_cm += linhas_confusao("distancia", i, o["y_dist"].to_numpy().astype(int), o[prob_cols("dist")].to_numpy().argmax(1), NOMES_DIST)

def sementes_sistema(i):
    return sorted(set(SEM["es"].get(CFG["es"][i], [])) & set(SEM["dist"].get(CFG["dist"][i], [])))

def pipeline_unidades(i, sementes):
    # Uma linha por imagem de teste × semente: p_es da rede (ou cabeça) E/S e p_dist da rede (ou cabeça) de distância.
    partes = []
    for s in sementes:
        for d in range(pc.N_DOBRAS):
            pe, pdi = PRED[(CFG["es"][i], s, d)], PRED[(CFG["dist"][i], s, d)]
            m = pe[["rel_path", "case_id", "y_es", "y_dist"] + prob_cols("es")].merge(
                pdi[["rel_path"] + prob_cols("dist")], on="rel_path", how="inner", validate="one_to_one")
            assert len(m) == len(pe) == len(pdi), f"{i} s{s} d{d}: as duas saídas não têm as mesmas imagens"
            partes.append(m.assign(semente=s, dobra=d))
    u = pd.concat(partes, ignore_index=True)
    u["y4"] = np.where(u["y_es"] == 1, 0, 1 + u["y_dist"])          # 0 saída, 1 encostado, 2 curta, 3 distância
    return u

def pred4(df):
    # a saída E/S decide; se for "entrada", vale a classe de distância
    return np.where(df[prob_cols("es")].to_numpy().argmax(1) == 1, 0, 1 + df[prob_cols("dist")].to_numpy().argmax(1))

def oof4(i, sementes):
    u = pipeline_unidades(i, sementes)
    o = u.groupby(["rel_path", "case_id", "y4", "dobra"])[prob_cols("es") + prob_cols("dist")].mean().reset_index().set_index("rel_path")
    return u, o

def f1_por_dobra4(u):
    f = {(s, d): f1_score(g["y4"], pred4(g), average="macro", labels=[0, 1, 2, 3], zero_division=0)
         for (s, d), g in u.groupby(["semente", "dobra"])}
    return pd.Series(f).groupby(level=1).mean()

PIPE, linhas_p = {}, []
for i in IDS:
    sem = sementes_sistema(i)
    if not sem:
        continue
    u, o = oof4(i, sem)
    y, p = o["y4"].to_numpy().astype(int), pred4(o)
    fd = f1_por_dobra4(u)
    PIPE[i] = {"sementes": sem, "f1_dobras": fd}
    rec = recall_score(y, p, average=None, labels=[0, 1, 2, 3], zero_division=0)
    linhas_p.append({"id": i, "sistema": NOMES[i], "sementes": ",".join(map(str, sem)),
                     "f1_macro_4": float(fd.mean()), "f1_macro_4_dp": float(fd.std(ddof=1)),
                     "f1_macro_4_oof": float(f1_score(y, p, average="macro", labels=[0, 1, 2, 3], zero_division=0)),
                     "acc_4_oof": float(accuracy_score(y, p)),
                     **{f"recall_{n}": float(rec[a]) for a, n in enumerate(NOMES4)}})
    linhas_cm += linhas_confusao("pipeline_4_classes", i, y, p, NOMES4)
TAB_P = pd.DataFrame(linhas_p)

def comparar_pipeline(a, b, n_boot=N_BOOT, semente=0):
    sem = sorted(set(PIPE[a]["sementes"]) & set(PIPE[b]["sementes"]))
    if not sem:
        return None
    ua, oa = oof4(a, sem)
    ub, ob = oof4(b, sem)
    assert set(oa.index) == set(ob.index), f"{a} × {b}: imagens diferentes no pipeline"
    ob = ob.loc[oa.index]
    y = oa["y4"].to_numpy().astype(int)
    pa, pb = pred4(oa), pred4(ob)
    bs = bootstrap_casos(oa["case_id"].to_numpy(), lambda am: f1_macro_rapido(y[am], pa[am], 4) - f1_macro_rapido(y[am], pb[am], 4),
                         n_boot, semente)
    dd = (f1_por_dobra4(ua) - f1_por_dobra4(ub)).dropna()
    return {"a": a, "b": b, "sementes": sem, "delta_f1_oof": f1_macro_rapido(y, pa, 4) - f1_macro_rapido(y, pb, 4),
            "ic95_f1": [float(x) for x in np.percentile(bs, [2.5, 97.5])], "n_boot_validos": int(len(bs)),
            "delta_f1_dobras": {int(i): round(float(v), 4) for i, v in dd.items()},
            "dobras_a_vence": int((dd > 0).sum()), "n_dobras": int(len(dd))}

COMP_P, linhas_cp = {}, []
for a, b in (("mt_kan", "st_kan"), ("mt_mlp", "st_mlp"), ("mt_kan", "mt_mlp")):
    if a in PIPE and b in PIPE:
        c = comparar_pipeline(a, b)
        COMP_P[f"pipeline__{a}_vs_{b}"] = c
        linhas_cp.append({"tarefa": "pipeline4", "contraste": f"{a} − {b}", "papel": "descritivo",
                          "sementes": ",".join(map(str, c["sementes"])), "ΔAUC OOF": None, "IC95 AUC": None,
                          "ΔF1 OOF": round(c["delta_f1_oof"], 4), "IC95 F1": [round(x, 4) for x in c["ic95_f1"]],
                          "a vence (dobras)": f"{c['dobras_a_vence']}/{c['n_dobras']}", "a não perde (dobras)": None,
                          "desfecho": "descritivo"})
if linhas_cp:
    TAB_C = pd.concat([TAB_C, pd.DataFrame(linhas_cp)], ignore_index=True)
TAB_CM = pd.DataFrame(linhas_cm)
display(TAB_P.round(4))
display(TAB_CM[TAB_CM.analise == "distancia"].round(3))
display(pd.DataFrame(COMP_P.values())[["a", "b", "sementes", "delta_f1_oof", "ic95_f1", "dobras_a_vence"]] if COMP_P else "sem pipeline ainda")

/tmp/ipykernel_1053/849097310.py:98: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  TAB_C = pd.concat([TAB_C, pd.DataFrame(linhas_cp)], ignore_index=True)


,id,sistema,sementes,f1_macro_4,f1_macro_4_dp,f1_macro_4_oof,acc_4_oof,recall_saida,recall_encostado,recall_curta,recall_distancia
0,st_kan,"Two single-task ResNet-152, KAN heads (P1 (d))","42,43,44",0.4998,0.0351,0.5151,0.7251,0.7556,0.1190,0.4784,0.7638
1,mt_kan,"One multitask ResNet-152, KAN heads","42,43,44",0.4906,0.0449,0.4848,0.7067,0.7884,0.0476,0.4397,0.7284
2,st_mlp,"Two single-task ResNet-152, MLP-128 heads (P1 ...","42,43,44",0.5059,0.0369,0.5142,0.7240,0.7541,0.1190,0.4871,0.7613
3,mt_mlp,"One multitask ResNet-152, MLP-128 heads","42,43,44",0.4994,0.0451,0.4918,0.7130,0.7884,0.0476,0.4612,0.7352


,analise,id,classe_real,n,recall,f1,pred_encostado,pred_curta,pred_distancia,pred_saida
0,distancia,st_kan,encostado,42,0.143,0.160,6,23,13,NaN
1,distancia,st_kan,curta,232,0.483,0.419,12,112,108,NaN
2,distancia,st_kan,distancia,1609,0.886,0.904,15,168,1426,NaN
3,distancia,mt_kan,encostado,42,0.143,0.164,6,21,15,NaN
4,distancia,mt_kan,curta,232,0.444,0.407,10,103,119,NaN
5,distancia,mt_kan,distancia,1609,0.897,0.906,15,150,1444,NaN
6,distancia,st_mlp,encostado,42,0.143,0.156,6,24,12,NaN
7,distancia,st_mlp,curta,232,0.491,0.421,11,114,107,NaN
8,distancia,st_mlp,distancia,1609,0.882,0.902,18,172,1419,NaN
9,distancia,mt_mlp,encostado,42,0.143,0.167,6,20,16,NaN


,a,b,sementes,delta_f1_oof,ic95_f1,dobras_a_vence
0,mt_kan,st_kan,"[42, 43, 44]",-0.030274,"[-0.06090484981865181, -0.0036737476006090427]",1
1,mt_mlp,st_mlp,"[42, 43, 44]",-0.022345,"[-0.05193239359915326, 0.0035587131482940466]",2
2,mt_kan,mt_mlp,"[42, 43, 44]",-0.007017,"[-0.01288227126235517, -0.0015914311141216632]",0


## 5. Custo: parâmetros, tempo de treino (rodadas na L4) e tempo de inferência (09)

In [9]:
def tempo_unidades(t, cfg):
    r = runs[t][(runs[t]["config"] == cfg) & ~runs[t]["reserva"]]
    return (pc.RECEITA["epocas_f1"] * r["f1_s_epoca"] + pc.RECEITA["epocas_f2"] * r["f2_s_epoca"]) / 60

def mem_unidades(t, cfg):
    r = runs[t][runs[t]["config"] == cfg]
    return [MEM.get((cfg, int(s), int(d)), float("nan")) for s, d in zip(r["semente"], r["dobra"])]

TB = mt.tabela_benchmark(bench) if bench else pd.DataFrame()
GPU_BENCH = (bench.get("gpu") or {}).get("nome") if bench else None
CUSTO = {}
for k in TIPOS:
    for i in (f"st_{k}", f"mt_{k}"):
        if i.startswith("mt"):
            cfg = mt.CONFIGS[k]
            tt = tempo_unidades("es", cfg)
            tm, ts = float(tt.mean()), float(tt.std(ddof=1)) if len(tt) > 1 else 0.0
            params, redes = mt.PARAMETROS_TOTAIS[cfg], 1
            cab = mt.PARAMETROS_CABECA[(k, "es")] + mt.PARAMETROS_CABECA[(k, "dist")]
            mem = np.nanmax(mem_unidades("es", cfg) or [np.nan])
            col_inf = "mt_ms_img"
        else:
            ce, cd = mt.CONFIGS_ST[k]["es"], mt.CONFIGS_ST[k]["dist"]
            te, td = tempo_unidades("es", ce), tempo_unidades("dist", cd)
            tm = float(te.mean() + td.mean())
            ts = float(np.sqrt((te.std(ddof=1) if len(te) > 1 else 0) ** 2 + (td.std(ddof=1) if len(td) > 1 else 0) ** 2))
            params, redes = mt.PARAMETROS_TOTAIS[ce] + mt.PARAMETROS_TOTAIS[cd], 2
            cab = mt.PARAMETROS_CABECA[(k, "es")] + mt.PARAMETROS_CABECA[(k, "dist")]
            mem = np.nanmax((mem_unidades("es", ce) + mem_unidades("dist", cd)) or [np.nan])
            col_inf = "par_st_ms_img"
        inf = {}
        for lote in (1, 32):
            x = TB[(TB.tipo == k) & (TB.lote == lote)] if len(TB) else TB
            inf[lote] = float(x[col_inf].iloc[0]) if len(x) else None
        CUSTO[i] = {"redes": redes, "params_total": int(params), "params_cabecas": int(cab),
                    "treinaveis_fase1": int(cab), "treinaveis_fase2": int(params),
                    "min_por_dobra": tm, "min_por_dobra_dp": ts, "mem_pico_treino_gb": float(mem),
                    "inf_ms_img_lote1": inf[1], "inf_ms_img_lote32": inf[32]}
    a, b = CUSTO[f"mt_{k}"], CUSTO[f"st_{k}"]
    a["reducao_params_pct"] = 100 * (a["params_total"] / b["params_total"] - 1)
    a["treino_mais_rapido_alem_do_ruido"] = bool(a["min_por_dobra"] + a["min_por_dobra_dp"] < b["min_por_dobra"] - b["min_por_dobra_dp"])
    for lote in (1, 32):
        x = TB[(TB.tipo == k) & (TB.lote == lote)] if len(TB) else TB
        a[f"inferencia_mais_rapida_lote{lote}"] = bool(x["mt_mais_rapida_alem_do_ruido"].iloc[0]) if len(x) else None
        a[f"par_st_sobre_mt_lote{lote}"] = float(x["par_st_sobre_mt"].iloc[0]) if len(x) else None
display(pd.DataFrame(CUSTO).T)
if len(TB):
    print("Tempo de inferência para as duas saídas (", GPU_BENCH, "|", bench.get("precisao"), "):")
    display(TB.round(3))

,redes,params_total,params_cabecas,treinaveis_fase1,treinaveis_fase2,min_por_dobra,min_por_dobra_dp,mem_pico_treino_gb,inf_ms_img_lote1,inf_ms_img_lote32,reducao_params_pct,treino_mais_rapido_alem_do_ruido,inferencia_mais_rapida_lote1,par_st_sobre_mt_lote1,inferencia_mais_rapida_lote32,par_st_sobre_mt_lote32
st_kan,2.0,124161536.0,7873920.0,7873920.0,124161536.0,14.947971,0.152025,3.626,51.034613,2.223557,NaN,NaN,NaN,NaN,NaN,NaN
mt_kan,1,66017728,7873920,7873920,66017728,8.583596,0.106646,3.719,27.312822,1.117195,-46.829163,True,True,1.868522,True,1.990304
st_mlp,2.0,116812805.0,525189.0,525189.0,116812805.0,14.581978,0.135806,3.526,48.484035,2.223767,NaN,NaN,NaN,NaN,NaN,NaN
mt_mlp,1,58668997,525189,525189,58668997,8.128131,0.09825,3.53,24.310445,1.110175,-49.775201,True,True,1.99437,True,2.003078


Tempo de inferência para as duas saídas ( NVIDIA L4 | amp_fp16 ):


,tipo,lote,mt_ms_img,mt_dp_ms_img,par_st_ms_img,par_st_dp_ms_img,par_st_sobre_mt,mt_mais_rapida_alem_do_ruido,mt_mem_pico_mb,par_st_mem_pico_mb,mt_parametros,par_st_parametros
0,kan,1,27.313,0.928,51.035,2.052,1.869,True,127.602,239.213,66017728,124161536
1,kan,32,1.117,0.011,2.224,0.014,1.990,True,162.674,281.662,66017728,124161536
2,mlp,1,24.310,1.258,48.484,0.962,1.994,True,122.319,235.306,58668997,116812805
3,mlp,32,1.110,0.013,2.224,0.016,2.003,True,162.674,287.255,58668997,116812805


## 6. Tabelas (CSV + LaTeX)

In [10]:
def pm(m, s, nd=3):
    return f"{m:.{nd}f} ± {s:.{nd}f}" if pd.notna(s) else (f"{m:.{nd}f}" if pd.notna(m) else "")

linhas = []
for i in IDS:
    linha = {"id": i, "sistema": NOMES[i], "redes": 2 if i.startswith("st") else 1}
    ok = all(i in RES[t].index for t in TAREFAS)
    if not ok:
        linhas.append({**linha, "status": "não executado"}); continue
    linha["status"] = "ok"
    linha.update({k: v for k, v in CUSTO[i].items() if k in ("params_total", "min_por_dobra", "mem_pico_treino_gb",
                                                              "inf_ms_img_lote1", "inf_ms_img_lote32")})
    linha["params_total_M"] = CUSTO[i]["params_total"] / 1e6
    for t in TAREFAS:
        r = RES[t].loc[i]
        for m in ("auc", "f1_macro", "acc", "acc_balanceada"):
            linha[f"{t}_{m}"] = pm(r[f"{t}_{m}_mean"], r[f"{t}_{m}_std"])
        linha[f"{t}_auc_oof"] = round(float(r["auc_oof"]), 4)
    pp = TAB_P[TAB_P.id == i]
    linha["pipeline_f1_macro_4"] = pm(pp["f1_macro_4"].iloc[0], pp["f1_macro_4_dp"].iloc[0]) if len(pp) else ""
    linha["pipeline_f1_macro_4_oof"] = round(float(pp["f1_macro_4_oof"].iloc[0]), 4) if len(pp) else None
    sem = sorted(set(RES["es"].loc[i, "sementes"]) & set(RES["dist"].loc[i, "sementes"]))
    linha["sementes"] = ",".join(map(str, sem))
    linha["reserva"] = bool(RES["es"].loc[i, "reserva"] or RES["dist"].loc[i, "reserva"])
    linhas.append(linha)
TAB = pd.DataFrame(linhas)
TAB.to_csv(R / "tabela_multitarefa.csv", index=False)
TAB_P.to_csv(R / "pipeline_multitarefa.csv", index=False)
TAB_CM.to_csv(R / "confusao_multitarefa.csv", index=False)
TAB_C.to_csv(R / "contrastes_multitarefa.csv", index=False)
display(TAB)

def tex_pm(s):
    return "$" + s.replace(" ± ", r" \pm ") + "$" if isinstance(s, str) and s else "--"

def sim(v):
    return False if v is None or (isinstance(v, float) and np.isnan(v)) else bool(v)

f = lambda v, nd: "--" if v is None or pd.isna(v) else f"{v:.{nd}f}"
sementes_txt = "; ".join(f"{r['id']}: {r['sementes']}".replace("_", chr(92) + "_") for _, r in TAB.iterrows() if r["status"] == "ok")
gpu_txt = GPU_BENCH.replace("NVIDIA ", "NVIDIA~") if GPU_BENCH else "--"
tex = [r"\begin{table*}[t]", r"\centering",
       r"\caption{Multitask vs.\ single-task under the single protocol (ResNet-152, grouped stratified 5-fold by case, same "
       r"two-phase recipe, last epoch). Single-task: two ResNet-152 trained separately, one per task (P1 heads (d) and (b)). "
       r"Multitask: one ResNet-152 with one head per task, trained jointly with $L = L_{\mathrm{E/S}} + L_{\mathrm{dist}}$ "
       r"(exits masked out of $L_{\mathrm{dist}}$). Mean $\pm$ s.d.\ across folds (each fold averaged over the training seeds); "
       r"AUC from predicted probabilities (distance: one-vs-rest macro, entries only); 4-class F1: macro F1 of the "
       r"exit/contact/close/distant pipeline. Training: minutes per fold on an NVIDIA~L4 ($15\,t_{F1}+45\,t_{F2}$; single-task = "
       r"sum of the two networks). Inference: ms per image to obtain both outputs (" + gpu_txt + r"). Seeds: " + sementes_txt
       + (r"; $\dagger$ = some units on another GPU" if any(sim(v) for v in TAB.get("reserva", [])) else "") + r".}",
       r"\label{tab:multitarefa}", r"\scriptsize", r"\setlength{\tabcolsep}{3pt}",
       r"\begin{tabular}{lcrcccccrrr}", r"\hline",
       r"System & Nets & Params (M) & E/S AUC & E/S F1 & Dist.\ AUC & Dist.\ F1 & 4-class F1 & Min/fold & ms/img (b=1) & ms/img (b=32) \\",
       r"\hline"]
for _, r in TAB.iterrows():
    nome = r["sistema"].replace("ResNet-152, ", "").replace("(P1 (d))", "(d)").replace("(P1 (b))", "(b)") + (r"$^\dagger$" if sim(r.get("reserva")) else "")
    if r["status"] != "ok":
        tex.append(f"{nome} & {r['redes']} & \\multicolumn{{9}}{{c}}{{not run}} \\\\"); continue
    tex.append(f"{nome} & {r['redes']} & {r['params_total_M']:.2f} & {tex_pm(r['es_auc'])} & {tex_pm(r['es_f1_macro'])} & "
               f"{tex_pm(r['dist_auc'])} & {tex_pm(r['dist_f1_macro'])} & {tex_pm(r['pipeline_f1_macro_4'])} & "
               f"{f(r['min_por_dobra'], 1)} & {f(r['inf_ms_img_lote1'], 2)} & {f(r['inf_ms_img_lote32'], 2)} \\\\")
tex += [r"\hline", r"\end{tabular}", r"\end{table*}"]
(R / "tabela_multitarefa.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")

ROT_TEX = {"es": "E/S", "dist": "Dist.", "pipeline4": "4-class"}
DESF_EN = {"melhora": "better", "nao_inferior": "non-inferior", "pior": "worse", "inconclusivo": "inconclusive"}
tex = [r"\begin{table}[t]", r"\centering",
       r"\caption{Paired contrasts (difference $a - b$; mt = multitask, st = single-task pair). $\Delta$ on the pooled "
       r"out-of-fold predictions (probabilities averaged over the common training seeds); 95\% CI from a case-level paired "
       r"bootstrap (" + str(N_BOOT) + r" resamples). Pre-registered non-inferiority margin: $" + f"{MARGEM_NI:.2f}" + r"$ on $\Delta$AUC.}",
       r"\label{tab:multitarefa_contrastes}", r"\scriptsize", r"\setlength{\tabcolsep}{3pt}",
       r"\begin{tabular}{llcccl}", r"\hline", r"Task & Contrast & $\Delta$AUC [95\% CI] & $\Delta$F1 [95\% CI] & $a$ wins & Outcome \\", r"\hline"]
for _, r in TAB_C[TAB_C["papel"] != "descritivo"].iterrows() if len(TAB_C) else []:
    tex.append(f"{ROT_TEX[r['tarefa']]} & {r['contraste'].replace('−', '$-$').replace('_', chr(92) + '_')} & "
               f"${sn(r['ΔAUC OOF'])}$ [${sn(r['IC95 AUC'][0])}$, ${sn(r['IC95 AUC'][1])}$] & "
               f"${sn(r['ΔF1 OOF'])}$ [${sn(r['IC95 F1'][0])}$, ${sn(r['IC95 F1'][1])}$] & {r['a vence (dobras)']} & "
               f"{DESF_EN[r['desfecho']]} \\\\")
for _, r in TAB_C[TAB_C["tarefa"] == "pipeline4"].iterrows() if len(TAB_C) else []:
    tex.append(f"4-class & {r['contraste'].replace('−', '$-$').replace('_', chr(92) + '_')} & -- & "
               f"${sn(r['ΔF1 OOF'])}$ [${sn(r['IC95 F1'][0])}$, ${sn(r['IC95 F1'][1])}$] & {r['a vence (dobras)']} & descriptive \\\\")
tex += [r"\hline", r"\end{tabular}", r"\end{table}"]
(R / "contrastes_multitarefa.tex").write_text("\n".join(tex) + "\n", encoding="utf-8")
print("Gravados: tabela_multitarefa.{csv,tex}, contrastes_multitarefa.{csv,tex}, pipeline_multitarefa.csv, confusao_multitarefa.csv")

,id,sistema,redes,status,params_total,min_por_dobra,mem_pico_treino_gb,inf_ms_img_lote1,inf_ms_img_lote32,params_total_M,...,es_auc_oof,dist_auc,dist_f1_macro,dist_acc,dist_acc_balanceada,dist_auc_oof,pipeline_f1_macro_4,pipeline_f1_macro_4_oof,sementes,reserva
0,st_kan,"Two single-task ResNet-152, KAN heads (P1 (d))",2,ok,124161536,14.947971,3.626,51.034613,2.223557,124.161536,...,0.9061,0.798 ± 0.058,0.477 ± 0.045,0.813 ± 0.017,0.515 ± 0.069,0.8054,0.500 ± 0.035,0.5151,"42,43,44",False
1,mt_kan,"One multitask ResNet-152, KAN heads",1,ok,66017728,8.583596,3.719,27.312822,1.117195,66.017728,...,0.8954,0.805 ± 0.065,0.482 ± 0.058,0.824 ± 0.021,0.511 ± 0.075,0.8127,0.491 ± 0.045,0.4848,"42,43,44",False
2,st_mlp,"Two single-task ResNet-152, MLP-128 heads (P1 ...",2,ok,116812805,14.581978,3.526,48.484035,2.223767,116.812805,...,0.9062,0.802 ± 0.062,0.482 ± 0.050,0.813 ± 0.013,0.520 ± 0.072,0.8106,0.506 ± 0.037,0.5142,"42,43,44",False
3,mt_mlp,"One multitask ResNet-152, MLP-128 heads",1,ok,58668997,8.128131,3.530,24.310445,1.110175,58.668997,...,0.8967,0.806 ± 0.071,0.492 ± 0.063,0.825 ± 0.018,0.520 ± 0.077,0.8148,0.499 ± 0.045,0.4918,"42,43,44",False


Gravados: tabela_multitarefa.{csv,tex}, contrastes_multitarefa.{csv,tex}, pipeline_multitarefa.csv, confusao_multitarefa.csv


## 7. Decisão e frases (EN para um eventual texto em inglês, PT para a dissertação)

In [11]:
def fmt(t, i, m="auc"):
    r = RES[t].loc[i]
    return f"{r[f'{t}_{m}_mean']:.3f} ± {r[f'{t}_{m}_std']:.3f}"

def mp(n):
    return f"{n / 1e6:.2f} M"

def ressalva(sem, en=True):
    if len(sem) < 2:
        return " (single training seed)" if en else " (uma única semente de treino)"
    return f" (seeds {', '.join(map(str, sem))})" if en else f" (sementes {', '.join(map(str, sem))})"

def ic(c, m="auc", en=True):
    lo, hi = c[f"ic95_{m}"]
    s = f"Δ{'AUC' if m == 'auc' else 'F1'} = {sn(c[f'delta_{m}_oof'])}, 95% CI [{sn(lo)}, {sn(hi)}]"
    return s if en else s.replace("95% CI", "IC 95 %")

FR_EN, FR_PT, DESFECHO = [], [], {}
for t in TAREFAS:
    for k in TIPOS:
        c = COMP.get(f"{t}__mt_{k}_vs_st_{k}")
        if c is None:
            continue
        d = c["desfecho"]; DESFECHO[f"{t}__{k}"] = d
        a, b = f"mt_{k}", f"st_{k}"
        num = f"AUC {fmt(t, a)} vs. {fmt(t, b)}; {ic(c)}; better in {c['dobras_a_vence']}/{c['n_dobras']} folds"
        num_pt = f"AUC {fmt(t, a)} contra {fmt(t, b)}; {ic(c, en=False)}; melhor em {c['dobras_a_vence']}/{c['n_dobras']} dobras"
        base = f"On {TASK_EN[t]}, under an identical protocol, one multitask ResNet-152 with {CABECA_EN[k]} (shared backbone, one head per task)"
        base_pt = f"Em {ROTULO[t]}, no mesmo protocolo, uma única ResNet-152 multitarefa com {CABECA_PT[k]} (backbone compartilhado, uma cabeça por tarefa)"
        ref_en = "the dedicated single-task network with the same head"
        ref_pt = "rede de tarefa única dedicada, com a mesma cabeça"
        margem_en = (f", although the difference stays within the pre-registered non-inferiority margin of {MARGEM_NI:+.2f}"
                     if c["dentro_da_margem"] else "")
        margem_pt = (f", embora a diferença fique dentro da margem de não inferioridade pré-registrada de {MARGEM_NI:+.2f}"
                     if c["dentro_da_margem"] else "")
        FR_EN.append(base + {
            "melhora": f" outperformed {ref_en} ({num}){ressalva(c['sementes'])}.",
            "nao_inferior": f" matched {ref_en} within the pre-registered non-inferiority margin of {MARGEM_NI:+.2f} ({num}; {ic(c, 'f1')}){ressalva(c['sementes'])}.",
            "pior": f" underperformed {ref_en} ({num}){margem_en}{ressalva(c['sementes'])}.",
            "inconclusivo": f" was not shown non-inferior to {ref_en} ({num}; the lower bound of the ΔAUC CI is below the pre-registered {MARGEM_NI:+.2f} margin), although the pre-registered rule for a difference (≥ 4/5 folds and CI excluding zero) was not met either{ressalva(c['sementes'])}.",
        }[d])
        FR_PT.append(base_pt + {
            "melhora": f" superou a {ref_pt} ({num_pt}){ressalva(c['sementes'], False)}.",
            "nao_inferior": f" foi não inferior à {ref_pt}, dentro da margem pré-registrada de {MARGEM_NI:+.2f} ({num_pt}; {ic(c, 'f1', False)}){ressalva(c['sementes'], False)}.",
            "pior": f" ficou abaixo da {ref_pt} ({num_pt}){margem_pt}{ressalva(c['sementes'], False)}.",
            "inconclusivo": f" não se mostrou não inferior à {ref_pt} ({num_pt}; o limite inferior do IC do ΔAUC fica abaixo da margem pré-registrada de {MARGEM_NI:+.2f}), mas a regra pré-registrada para uma diferença (≥ 4/5 dobras e IC sem o zero) também não foi cumprida{ressalva(c['sementes'], False)}.",
        }[d])
    c = COMP.get(f"{t}__mt_kan_vs_mt_mlp")
    if c is not None:
        hk = mt.PARAMETROS_CABECA[("kan", "es")] + mt.PARAMETROS_CABECA[("kan", "dist")]
        hm = mt.PARAMETROS_CABECA[("mlp", "es")] + mt.PARAMETROS_CABECA[("mlp", "dist")]
        num = f"AUC {fmt(t, 'mt_kan')} vs. {fmt(t, 'mt_mlp')}; {ic(c)}; KAN better in {c['dobras_a_vence']}/{c['n_dobras']} folds"
        num_pt = f"AUC {fmt(t, 'mt_kan')} contra {fmt(t, 'mt_mlp')}; {ic(c, en=False)}; KAN melhor em {c['dobras_a_vence']}/{c['n_dobras']} dobras"
        sem_en, sem_pt = ressalva(c["sementes"]), ressalva(c["sementes"], False)
        mlp_en, mlp_pt = f"MLP-128 heads ({mp(hm)})", f"cabeças MLP-128 ({mp(hm)})"
        cab_en = f"Within the multitask network, on {TASK_EN[t]}, spline-KAN heads ({mp(hk)} head parameters)"
        cab_pt = f"Dentro da rede multitarefa, em {ROTULO[t]}, as cabeças KAN spline ({mp(hk)} parâmetros nas cabeças)"
        FR_EN.append(cab_en + {
            "melhora": f" outperformed {mlp_en} ({num}){sem_en}.",
            "nao_inferior": f" performed on par with {mlp_en}, within the pre-registered margin ({num}){sem_en}.",
            "pior": f" underperformed {mlp_en} ({num}){sem_en}.",
            "inconclusivo": f" were neither shown non-inferior to nor different from {mlp_en} ({num}){sem_en}.",
        }[c["desfecho"]] + " This compares head types, not parameter efficiency.")
        FR_PT.append(cab_pt + {
            "melhora": f" superaram as {mlp_pt} ({num_pt}){sem_pt}.",
            "nao_inferior": f" empataram com as {mlp_pt}, dentro da margem pré-registrada ({num_pt}){sem_pt}.",
            "pior": f" ficaram abaixo das {mlp_pt} ({num_pt}){sem_pt}.",
            "inconclusivo": f" não se mostraram nem não inferiores nem diferentes das {mlp_pt} ({num_pt}){sem_pt}.",
        }[c["desfecho"]] + " Isso compara tipos de cabeça, não eficiência em parâmetros.")

for k in TIPOS:
    a, b = CUSTO.get(f"mt_{k}"), CUSTO.get(f"st_{k}")
    if not a or not b or f"mt_{k}" not in RES["es"].index:
        continue
    gpu = GPU_BENCH or "the GPU"
    txt_en = (f"One multitask network with {CABECA_EN[k]} replaces two single-task networks: {mp(a['params_total'])} vs. "
              f"{mp(b['params_total'])} parameters ({a['reducao_params_pct']:+.1f}%).")
    txt_pt = (f"Uma rede multitarefa com {CABECA_PT[k]} substitui duas redes de tarefa única: {mp(a['params_total'])} contra "
              f"{mp(b['params_total'])} parâmetros ({a['reducao_params_pct']:+.1f} %).")
    if a["inf_ms_img_lote32"] is not None:
        if a["inferencia_mais_rapida_lote32"]:
            txt_en += (f" Both outputs need one forward pass instead of two: {a['inf_ms_img_lote32']:.2f} vs. {b['inf_ms_img_lote32']:.2f} ms "
                       f"per image at batch 32 and {a['inf_ms_img_lote1']:.2f} vs. {b['inf_ms_img_lote1']:.2f} ms at batch 1 on an {gpu} "
                       f"({a['par_st_sobre_mt_lote32']:.1f}× faster at batch 32).")
            txt_pt += (f" As duas saídas saem de um forward em vez de dois: {a['inf_ms_img_lote32']:.2f} contra {b['inf_ms_img_lote32']:.2f} ms "
                       f"por imagem com lote 32 e {a['inf_ms_img_lote1']:.2f} contra {b['inf_ms_img_lote1']:.2f} ms com lote 1 na {gpu} "
                       f"({a['par_st_sobre_mt_lote32']:.1f}× mais rápido com lote 32).")
        else:
            txt_en += f" Inference for both outputs is not faster beyond measurement noise ({a['inf_ms_img_lote32']:.2f} vs. {b['inf_ms_img_lote32']:.2f} ms per image at batch 32)."
            txt_pt += f" A inferência das duas saídas não fica mais rápida além do ruído ({a['inf_ms_img_lote32']:.2f} contra {b['inf_ms_img_lote32']:.2f} ms por imagem com lote 32)."
    txt_en += (f" Training takes {a['min_por_dobra']:.1f} min per fold for the multitask network vs. {b['min_por_dobra']:.1f} min for the two "
               f"single-task networks together (NVIDIA L4" + ("; faster beyond run-to-run noise)." if a["treino_mais_rapido_alem_do_ruido"] else ")."))
    txt_pt += (f" O treino leva {a['min_por_dobra']:.1f} min por dobra para a rede multitarefa contra {b['min_por_dobra']:.1f} min para as duas "
               f"redes de tarefa única juntas (L4" + ("; mais rápido além da variação entre rodadas)." if a["treino_mais_rapido_alem_do_ruido"] else ")."))
    FR_EN.append(txt_en); FR_PT.append(txt_pt)

for (a, b) in (("mt_kan", "st_kan"), ("mt_mlp", "st_mlp")):
    c = COMP_P.get(f"pipeline__{a}_vs_{b}")
    if c is None:
        continue
    fa, fb = (float(TAB_P.loc[TAB_P.id == x, "f1_macro_4_oof"].iloc[0]) for x in (a, b))
    k = a.split("_")[1]
    FR_EN.append(f"Secondary, descriptive: in the 4-class pipeline (exit, contact, close, distant) with {CABECA_EN[k]}, macro-F1 on the pooled "
                 f"out-of-fold predictions is {fa:.3f} for the multitask network vs. {fb:.3f} for the single-task pair ({ic(c, 'f1')}).")
    FR_PT.append(f"Secundário, descritivo: no pipeline de 4 classes (saída, encostado, curta, distância) com {CABECA_PT[k]}, o F1 macro no OOF "
                 f"é {fa:.3f} para a rede multitarefa contra {fb:.3f} para o par de tarefa única ({ic(c, 'f1', False)}).")

pode = {
    "mt_non_inferior_to_st": {k: v in ("melhora", "nao_inferior") for k, v in DESFECHO.items()},
    "mt_better_than_st": {k: v == "melhora" for k, v in DESFECHO.items()},
    "mt_worse_than_st": {k: v == "pior" for k, v in DESFECHO.items()},
    "kan_better_than_mlp_within_mt": {t: COMP[f"{t}__mt_kan_vs_mt_mlp"]["desfecho"] == "melhora"
                                      for t in TAREFAS if f"{t}__mt_kan_vs_mt_mlp" in COMP},
    "one_network_instead_of_two": True,
    "mt_faster_inference": {f"{k}_lote{l}": CUSTO[f"mt_{k}"].get(f"inferencia_mais_rapida_lote{l}")
                            for k in TIPOS for l in (1, 32) if f"mt_{k}" in CUSTO},
    "mt_faster_training": {k: CUSTO[f"mt_{k}"]["treino_mais_rapido_alem_do_ruido"] for k in TIPOS if f"mt_{k}" in CUSTO},
    "kan_head_parameter_efficiency": False,
    "claims_about_contact_shots": False,
    "compare_with_0.908_or_0.893": False,
    "spline_KAN_is_the_reference_that_works": False,
}
decisao = {"margem_ni": MARGEM_NI, "n_boot": N_BOOT, "desfecho_mt_vs_st": DESFECHO, "contrastes": COMP, "pipeline": COMP_P,
           "custo": CUSTO, "benchmark_inferencia": {"gpu": GPU_BENCH, "precisao": bench.get("precisao") if bench else None,
                                                    "tabela": TB.to_dict("records") if len(TB) else []},
           "pode_afirmar": pode, "sementes_por_config": {t: {ID[t][c]: s for c, s in SEM[t].items()} for t in TAREFAS},
           "avisos": avisos, "frases_en": FR_EN, "frases_pt": FR_PT,
           "plugin_sha256": SHA_PLUGIN_ESPERADO, "protocolo_sha256": SHA_PROTOCOLO_ESPERADO}
pc.gravar_json_atomico(decisao, R / "decisao_multitarefa.json")
texto = ("# Frases da multitarefa (geradas pelo 10)\n\n## EN\n\n" + "\n\n".join(f"- {x}" for x in FR_EN)
         + "\n\n## PT (dissertação)\n\n" + "\n\n".join(f"- {x}" for x in FR_PT)
         + f"\n\n## Pode afirmar\n\n```json\n{json.dumps(pode, indent=1, ensure_ascii=False)}\n```\n"
         + "\n## Lembretes\n\n"
         + "- O resultado entra só no TCC; o artigo e o pôster do SIBGRAPI mantêm a multitarefa como trabalho futuro.\n"
         + "- O argumento é uma rede no lugar de duas, com os parâmetros contados; nada de \"parameter efficiency\" da cabeça KAN.\n"
         + "- Distância: prever sempre \"distância\" já dá acurácia 0,854 (F1 macro 0,307); reporte AUC e F1 macro, não só a acurácia.\n"
         + "- Nada de afirmação específica sobre encostado: são 16 casos (42 imagens), ~8 imagens por dobra.\n"
         + "- Não comparar com 0,908 (TCC antigo) nem com 0,893 (artigo do SIBGRAPI): protocolos diferentes.\n"
         + "- Na MT, a cabeça E/S vê 2.554 imagens e a de distância 1.883; a ST de distância do P1 treinou só nas entradas. A avaliação é pareada nas mesmas imagens.\n"
         + "- GaussNoise(std_range=(0,2; 0,44)) \"como rodou\" pode apagar a tatuagem e prejudicar a classe curta; vale igualmente para ST e MT (limitação, não viés da comparação).\n"
         + "- Citar Lira et al. (2025, IJLM) como origem da base; a base não tem licença de redistribuição e nenhuma imagem sai da sessão.\n")
(R / "frases_multitarefa.md").write_text(texto, encoding="utf-8")
print(texto)

# Frases da multitarefa (geradas pelo 10)

## EN

- On entry/exit classification, under an identical protocol, one multitask ResNet-152 with spline-KAN heads (shared backbone, one head per task) underperformed the dedicated single-task network with the same head (AUC 0.894 ± 0.014 vs. 0.905 ± 0.013; ΔAUC = -0.011, 95% CI [-0.015, -0.006]; better in 0/5 folds), although the difference stays within the pre-registered non-inferiority margin of -0.02 (seeds 42, 43, 44).

- On entry/exit classification, under an identical protocol, one multitask ResNet-152 with MLP-128 heads (shared backbone, one head per task) underperformed the dedicated single-task network with the same head (AUC 0.895 ± 0.014 vs. 0.905 ± 0.014; ΔAUC = -0.010, 95% CI [-0.014, -0.005]; better in 0/5 folds), although the difference stays within the pre-registered non-inferiority margin of -0.02 (seeds 42, 43, 44).

- Within the multitask network, on entry/exit classification, spline-KAN heads (7.87 M head parameters) perfo